## Этот ноутбук будет посвещен выбору признаков для построенния модели 

#### План:
- Удаление константных и квазиконстантных признаков
- Удаление признаков с критическим объемом пропусков
- Features enginеering
- Борьба с мультиколлинеарностью
- Mutual Information
- RFECV
- Xgboost features importance


In [ ]:
from sklearn.feature_selection import mutual_info_regression
from sklearn.feature_selection import RFECV
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from lightgbm import LGBMRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from catboost import CatBoostRegressor, Pool
import pandas as pd
import pandas as pd
import xgboost
import warnings
warnings.filterwarnings("ignore")

In [7]:
df = pd.read_csv('../data/cleaned_EDA.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
test_df = pd.read_csv('../data/test.csv', index_col='id')
test_df['timestamp'] = pd.to_datetime(test_df['timestamp'])
macro = pd.read_csv('../data/macro.csv')
macro['timestamp'] = pd.to_datetime(macro['timestamp'])
test_df = test_df.merge(macro, on = 'timestamp', how = 'left')
test_df

,timestamp,full_sq,life_sq,floor,max_floor,material,build_year,num_room,kitch_sq,state,...,provision_retail_space_modern_sqm,turnover_catering_per_cap,theaters_viewers_per_1000_cap,seats_theather_rfmin_per_100000_cap,museum_visitis_per_100_cap,bandwidth_sports,population_reg_sports_share,students_reg_sports_share,apartment_build,apartment_fund_sqm
0,2015-07-01,39.00,20.7,2,9,1,1998.0,1,8.9,3.0,...,NaN,10805.0,NaN,0.45888,NaN,463938.0,NaN,NaN,NaN,234576.9
1,2015-07-01,79.20,NaN,8,17,1,0.0,3,1.0,1.0,...,NaN,10805.0,NaN,0.45888,NaN,463938.0,NaN,NaN,NaN,234576.9
2,2015-07-01,40.50,25.1,3,5,2,1960.0,2,4.8,2.0,...,NaN,10805.0,NaN,0.45888,NaN,463938.0,NaN,NaN,NaN,234576.9
3,2015-07-01,62.80,36.0,17,17,1,2016.0,2,62.8,3.0,...,NaN,10805.0,NaN,0.45888,NaN,463938.0,NaN,NaN,NaN,234576.9
4,2015-07-01,40.00,40.0,17,17,1,0.0,1,1.0,1.0,...,NaN,10805.0,NaN,0.45888,NaN,463938.0,NaN,NaN,NaN,234576.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7657,2016-05-26,52.20,31.8,10,12,5,1973.0,2,9.1,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7658,2016-05-28,54.09,NaN,14,0,1,NaN,2,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7659,2016-05-30,41.08,1.0,12,1,1,1.0,1,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7660,2016-05-30,34.80,19.8,8,9,5,1977.0,1,6.4,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30461 entries, 0 to 30460
Columns: 391 entries, Unnamed: 0 to apartment_fund_sqm
dtypes: datetime64[us](1), float64(219), int64(156), str(15)
memory usage: 90.9 MB


In [9]:

df['rel_floor'] = df['floor'] / df['max_floor']
df['kitchen_ratio'] = df['kitch_sq'] / df['full_sq']
df['living_ratio'] = df['life_sq'] / df['full_sq']
df['house_age'] = df['timestamp'].dt.year - df['build_year'] 
df['year'] = df['timestamp'].dt.year
df['month'] = df['timestamp'].dt.month
df['day'] = df['timestamp'].dt.day
df['dayofweek'] = df['timestamp'].dt.dayofweek
df['hour'] = df['timestamp'].dt.hour

In [10]:
df[['rel_floor', 'kitchen_ratio', 'living_ratio', 'house_age']]

,rel_floor,kitchen_ratio,living_ratio,house_age
0,NaN,NaN,0.627907,NaN
1,NaN,NaN,0.558824,NaN
2,NaN,NaN,0.674419,NaN
3,NaN,NaN,0.561798,NaN
4,NaN,NaN,1.000000,NaN
...,...,...,...,...
30456,0.777778,0.136364,0.613636,40.0
30457,0.333333,0.116279,0.686047,80.0
30458,0.500000,0.022222,NaN,NaN
30459,0.333333,0.171875,0.500000,12.0


In [11]:
trash_features = []
X = df.drop('price_doc', axis=1)
y = df['price_doc']


In [12]:
numeric_cols = X.select_dtypes(include='number').columns

selector = VarianceThreshold(threshold=0.01)
X_numeric = selector.fit_transform(X[numeric_cols])

mask = selector.get_support()

removed_features = numeric_cols[~mask]

X = X.drop(columns=removed_features)

removed_features

Index(['mosque_count_500', 'gdp_annual_growth', 'deposits_growth',
       'grp_growth', 'real_dispos_income_per_cap_growth', 'salary_growth',
       'unemployment', 'employment', 'profitable_enterpr_share',
       'unprofitable_enterpr_share', 'share_own_revenues', 'divorce_rate',
       'lodging_sqm_per_cap', 'heating_share', 'old_house_share',
       'provision_retail_space_modern_sqm',
       'seats_theather_rfmin_per_100000_cap', 'kitchen_ratio', 'hour'],
      dtype='str')

In [13]:
df_na_sum = X.isna().sum().reset_index().rename(columns={0:'NA_sum', 'index':'feature'}).sort_values('NA_sum', ascending=False).head(40)
df_na_sum['norm'] = df_na_sum['NA_sum'] / len(X) *100
print(df_na_sum.head(5))
na_trash = df_na_sum[df_na_sum['norm']>=80]['feature'].to_list()
X = X.drop(na_trash, axis=1)

                                    feature  NA_sum       norm
364              provision_retail_space_sqm   24871  81.648666
366           theaters_viewers_per_1000_cap   16896  55.467647
352  load_of_teachers_preschool_per_teacher   16896  55.467647
370               students_reg_sports_share   16896  55.467647
367              museum_visitis_per_100_cap   16896  55.467647


In [14]:
numeric_cols = X.select_dtypes(include='number').columns
X_no_na = X[numeric_cols].dropna()
y_aligned = y.loc[X_no_na.index]

mi = mutual_info_regression(X_no_na, y_aligned)

mi_df = pd.DataFrame({
    'feature': X_no_na.columns,
    'mutual_info': mi
})

mi_df.sort_values(
    'mutual_info',
    ascending=False
).head(10)

,feature,mutual_info
1,full_sq,0.513143
2,life_sq,0.383999
7,num_room,0.221257
358,house_age,0.185477
6,build_year,0.180971
8,kitch_sq,0.163618
87,ID_railroad_station_walk,0.141933
62,build_count_panel,0.128409
254,office_sqm_5000,0.122067
249,sport_count_3000,0.116636


In [15]:
X2 = X.drop(columns=['timestamp'])
cat_features = X2.select_dtypes(include='object').columns

preprocess = ColumnTransformer([
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_features)
], remainder='passthrough')

X_encoded = preprocess.fit_transform(X2)

lgbm = LGBMRegressor(n_estimators=100, n_jobs=1, random_state=42)

rfecv_lgbm = RFECV(
    estimator=lgbm,
    step=5,
    cv=KFold(3, shuffle=True, random_state=42),
    scoring="neg_root_mean_squared_log_error",
    n_jobs=-1
)

rfecv_lgbm.fit(X_encoded, y)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.322073 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 45104
[LightGBM] [Info] Number of data points in the train set: 30461, number of used features: 378
[LightGBM] [Info] Start training from score 7123676.293687
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.297778 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 45090
[LightGBM] [Info] Number of data points in the train set: 30461, number of used features: 373
[LightGBM] [Info] Start training from score 7123676.293687
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.105244 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 44783
[LightGBM] [Info] Number of data points

,estimator estimator: ``Estimator`` instanceA supervised learning estimator with a ``fit`` method that providesinformation about feature importance either through a ``coef_``attribute or through a ``feature_importances_`` attribute.,LGBMRegressor...ndom_state=42)
,"step step: int or float, default=1If greater than or equal to 1, then ``step`` corresponds to the(integer) number of features to remove at each iteration.If within (0.0, 1.0), then ``step`` corresponds to the percentage(rounded down) of features to remove at each iteration.Note that the last iteration may remove fewer than ``step`` features inorder to reach ``min_features_to_select``.",5
,"min_features_to_select min_features_to_select: int, default=1The minimum number of features to be selected. This number of featureswill always be scored, even if the difference between the originalfeature count and ``min_features_to_select`` isn't divisible by``step``... versionadded:: 0.20",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If theestimator is not a classifier or if ``y`` is neither binary nor multiclass,:class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value of None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"scoring scoring: str or callable, default=NoneScoring method to evaluate the :class:`RFE` selectors' performance. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: the `estimator`'s :ref:`default evaluation criterion ` is used.",'neg_root_mean_squared_log_error'
,"verbose verbose: int, default=0Controls verbosity of output.",0
,"n_jobs n_jobs: int or None, default=NoneNumber of cores to run in parallel while fitting across folds.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionadded:: 0.18",-1
,"importance_getter importance_getter: str or callable, default='auto'If 'auto', uses the feature importance either through a `coef_`or `feature_importances_` attributes of estimator.Also accepts a string that specifies an attribute name/pathfor extracting feature importance.For example, give `regressor_.coef_` in case of:class:`~sklearn.compose.TransformedTargetRegressor` or`named_steps.clf.feature_importances_` in case of:class:`~sklearn.pipeline.Pipeline` with its last step named `clf`.If `callable`, overrides the default feature importance getter.The callable is passed with the fitted estimator and it shouldreturn importance for each feature... versionadded:: 0.24",'auto'
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1


In [16]:
print(f"Было признаков: {X_encoded.shape[1]}")
print(f"Осталось после отбора: {rfecv_lgbm.n_features_}")

Было признаков: 378
Осталось после отбора: 148


In [17]:
all_feature_names = preprocess.get_feature_names_out()

selected_features = all_feature_names[rfecv_lgbm.support_]

print("Топ-30 оставшихся признаков:")
print(selected_features[:30]) 

Топ-30 оставшихся признаков:
['cat__product_type' 'cat__sub_area' 'cat__culture_objects_top_25'
 'remainder__Unnamed: 0' 'remainder__full_sq' 'remainder__life_sq'
 'remainder__floor' 'remainder__max_floor' 'remainder__material'
 'remainder__build_year' 'remainder__num_room' 'remainder__kitch_sq'
 'remainder__state' 'remainder__green_zone_part' 'remainder__indust_part'
 'remainder__preschool_quota' 'remainder__school_quota'
 'remainder__hospital_beds_raion' 'remainder__sport_objects_raion'
 'remainder__full_all' 'remainder__ekder_male'
 'remainder__raion_build_count_with_material_info'
 'remainder__build_count_brick' 'remainder__build_count_monolith'
 'remainder__build_count_1946-1970' 'remainder__ID_metro'
 'remainder__metro_min_avto' 'remainder__metro_km_avto'
 'remainder__metro_min_walk' 'remainder__kindergarten_km']


In [18]:
best = rfecv_lgbm.cv_results_['mean_test_score'].max() * -1
print(f"Лучший средний rmsle на кросс-валидации: {best:.4f}")

Лучший средний rmsle на кросс-валидации: 0.4676


In [44]:
X_clean = X.drop('timestamp', axis=1).copy()

cat_features = X_clean.select_dtypes(include=['object', 'category']).columns.tolist()

X_clean[cat_features] = X_clean[cat_features].fillna("Missing")

train_pool = Pool(X_clean, y, cat_features=cat_features)
model = CatBoostRegressor(iterations=1000, random_state=42)

print("Запускаем отбор признаков CatBoost...")
summary = model.select_features(
    train_pool,
    eval_set=train_pool,       
    features_for_select=list(range(X_clean.shape[1])),
    num_features_to_select=100, 
    steps=3,                   
    logging_level='Silent'
)

print("Отобранные признаки:", summary['selected_features_names'])

Запускаем отбор признаков CatBoost...
Отобранные признаки: ['Unnamed: 0', 'full_sq', 'life_sq', 'floor', 'max_floor', 'build_year', 'num_room', 'kitch_sq', 'state', 'product_type', 'sub_area', 'area_m', 'indust_part', 'school_education_centers_raion', 'hospital_beds_raion', 'culture_objects_top_25', 'build_count_monolith', 'ID_metro', 'metro_min_avto', 'metro_km_avto', 'metro_min_walk', 'kindergarten_km', 'school_km', 'park_km', 'green_zone_km', 'industrial_km', 'cemetery_km', 'ID_railroad_station_walk', 'railroad_station_avto_km', 'public_transport_station_km', 'water_km', 'ttk_km', 'sadovoe_km', 'big_road1_km', 'big_road2_km', 'railroad_km', 'nuclear_reactor_km', 'radiation_km', 'power_transmission_line_km', 'thermal_power_plant_km', 'ts_km', 'market_shop_km', 'fitness_km', 'swim_pool_km', 'ice_rink_km', 'stadium_km', 'basketball_km', 'hospice_morgue_km', 'detention_facility_km', 'public_healthcare_km', 'university_km', 'workplaces_km', 'shopping_centers_km', 'office_km', 'additional

In [45]:
len(summary['selected_features_names'])

100